In [4]:
from pathlib import Path
import shutil
import random

src = Path(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped")
dst = Path(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped_dataset")

train_ratio = 0.8
image_exts = {".jpg", ".jpeg", ".png", ".webp"}

random.seed(42)

for class_folder in src.iterdir():
    if not class_folder.is_dir():
        continue

    images = [p for p in class_folder.iterdir() if p.suffix.lower() in image_exts]
    random.shuffle(images)

    split_idx = int(len(images) * train_ratio)
    train_images = images[:split_idx]
    val_images = images[split_idx:]

    for split_name, split_images in [("train", train_images), ("val", val_images)]:
        out_folder = dst / split_name / class_folder.name
        out_folder.mkdir(parents=True, exist_ok=True)

        for img in split_images:
            shutil.copy2(img, out_folder / img.name)

print("Done. New dataset created at:", dst)

Done. New dataset created at: C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped_dataset


In [5]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")

model.train(
    data=r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped_dataset",
    epochs=50,
    imgsz=224,
    batch=16,
    project="material_model_runs",
    name="starter_classifier"
)

Ultralytics 8.4.84  Python-3.13.14 torch-2.12.1+cpu CPU (Intel Core Ultra 9 185H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=starter_classifier-2, nbs=

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A5E3FCEF90>
curves: []
curves_results: []
fitness: 0.8333333432674408
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.6666666865348816, 'metrics/accuracy_top5': 1.0, 'fitness': 0.8333333432674408}
save_dir: WindowsPath('C:/Users/pstanam.MRX/Downloads/OneDrive_2026-02-23/McCraken Ticket System/runs/classify/material_model_runs/starter_classifier-2')
speed: {'preprocess': 0.0017381029292231516, 'inference': 3.307838092691132, 'loss': 5.2375807648613345e-05, 'postprocess': 0.00015238211268470402}
top1: 0.6666666865348816
top5: 1.0

In [10]:
import cv2
from ultralytics import YOLO

model = YOLO(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\runs\classify\material_model_runs\starter_classifier-2\weights\best.pt")

img = cv2.imread(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\material_cropped\Other Clean Hard Fill (rubble & mixed material)\Truck_image_2026-06-29T08-54-46_DT-2026-011330.jpeg")

crop = img  # adjust later

results = model.predict(crop)

for r in results:
    class_id = r.probs.top1
    confidence = r.probs.top1conf.item()
    print(model.names[class_id], confidence)


0: 224x224 Other Clean Hard Fill (rubble & mixed material) 0.91, Broken Concrete 0.08, Broken Asphalt (or material mixed asphalt) 0.01, Slop 0.00, 24.8ms
Speed: 7.3ms preprocess, 24.8ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Other Clean Hard Fill (rubble & mixed material) 0.9114165902137756
